# Fine-tuning InLegalBERT for rental-agreement risk detection

This notebook fine-tunes **InLegalBERT** (a BERT model pre-trained on Indian legal text) to detect **risky clauses**, and compares it with the project's rules and the TF-IDF classifier under the **same agreement-grouped 5-fold cross-validation** used in the paper.

**How to run**
1. *Runtime → Change runtime type → T4 GPU* (free).
2. *Runtime → Run all*.
3. When asked, upload **`bert_data.csv`** (from the project's `data` folder).
4. Wait ~10–20 minutes. Results tables appear at the end, and `bert_results.md` + `bert_predictions.csv` download automatically.

Nothing here needs editing. The settings cell below is optional.

In [ ]:
!pip -q install transformers scikit-learn pandas openpyxl

## Settings (optional)

In [ ]:
import os
MODEL_NAME = os.environ.get("MODEL_NAME", "law-ai/InLegalBERT")  # alternatives: "nlpaueb/legal-bert-base-uncased", "bert-base-uncased"
EPOCHS     = int(os.environ.get("EPOCHS", 4))
MAX_LEN    = int(os.environ.get("MAX_LEN", 256))
BATCH      = 16
LR         = 2e-5
USE_EXTRA  = True      # also train on the authored + synthetic clauses (never tested on)
THRESHOLD  = 0.5       # fixed in advance; not tuned on test folds
FOLDS      = 5
SEED       = 42
DATA_FILE  = os.environ.get("DATA_FILE", "bert_data.csv")

## Load data

In [ ]:
import pandas as pd, numpy as np
if not os.path.exists(DATA_FILE):
    from google.colab import files
    print("Upload bert_data.csv from the project's data folder")
    files.upload()
df = pd.read_csv(DATA_FILE)
real  = df[df.split == "real"].reset_index(drop=True)
extra = df[df.split == "extra"].reset_index(drop=True)
real["y"]  = (real.risk == "RISKY").astype(int)
extra["y"] = (extra.risk == "RISKY").astype(int)
print(f"real clauses: {len(real)} from {real.agreement.nunique()} agreements "
      f"({(real.groupby('agreement').region.first()=='india').sum()} Indian) | risky: {real.y.sum()}")
print(f"extra training clauses: {len(extra)}")

## Model and training loop

In [ ]:
import torch, random
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| model:", MODEL_NAME)
if device == "cpu":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, or this will be very slow.")
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

class Clauses(Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tok(list(texts), truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt")
        self.labels = None if labels is None else torch.tensor(list(labels))
    def __len__(self): return self.enc["input_ids"].shape[0]
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None: item["labels"] = self.labels[i]
        return item

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def train_and_predict(train_texts, train_y, test_texts):
    set_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
    dl = DataLoader(Clauses(train_texts, train_y), batch_size=BATCH, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * len(dl)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    # class weights: risky clauses are rare
    n_pos = max(1, int(np.sum(train_y))); n_neg = len(train_y) - n_pos
    weights = torch.tensor([1.0, n_neg / n_pos], dtype=torch.float, device=device)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
    model.train()
    for ep in range(EPOCHS):
        total = 0.0
        for b in dl:
            b = {k: v.to(device) for k, v in b.items()}
            labels = b.pop("labels")
            loss = loss_fn(model(**b).logits, labels)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad(); total += loss.item()
        print(f"    epoch {ep+1}/{EPOCHS} loss {total/len(dl):.4f}")
    model.eval(); probs = []
    with torch.no_grad():
        for b in DataLoader(Clauses(test_texts), batch_size=64):
            b = {k: v.to(device) for k, v in b.items()}
            probs += torch.softmax(model(**b).logits, -1)[:, 1].cpu().tolist()
    del model; torch.cuda.empty_cache()
    return np.array(probs)

## Agreement-grouped 5-fold cross-validation

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression

bert_prob = np.zeros(len(real)); tfidf_prob = np.zeros(len(real)); fold_of = np.zeros(len(real), int)
for k, (tr, te) in enumerate(GroupKFold(n_splits=FOLDS).split(real, real.y, real.agreement)):
    tr_df = pd.concat([real.iloc[tr], extra]) if USE_EXTRA else real.iloc[tr]
    print(f"fold {k+1}/{FOLDS}: train {len(tr_df)} clauses, test {len(te)} clauses "
          f"from {real.iloc[te].agreement.nunique()} unseen agreements")
    bert_prob[te] = train_and_predict(tr_df.text.tolist(), tr_df.y.tolist(), real.text.iloc[te].tolist())
    # TF-IDF + logistic regression baseline on exactly the same split
    feats = FeatureUnion([("w", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, lowercase=True)),
                          ("c", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True))])
    Xtr = feats.fit_transform(tr_df.text); Xte = feats.transform(real.text.iloc[te])
    lr = LogisticRegression(max_iter=3000, C=8.0, class_weight="balanced").fit(Xtr, tr_df.y)
    tfidf_prob[te] = lr.predict_proba(Xte)[:, 1]
    fold_of[te] = k

## Results

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
rules = (real.rule_pred == "RISKY").values
preds = {
    "Rules": rules,
    "TF-IDF + LR": tfidf_prob >= THRESHOLD,
    "Hybrid: rules + TF-IDF": rules | (tfidf_prob >= THRESHOLD),
    f"{MODEL_NAME.split('/')[-1]}": bert_prob >= THRESHOLD,
    f"Hybrid: rules + {MODEL_NAME.split('/')[-1]}": rules | (bert_prob >= THRESHOLD),
}
lines = [f"# Risky-clause detection, agreement-grouped {FOLDS}-fold CV",
         f"Model: {MODEL_NAME} | epochs {EPOCHS} | max_len {MAX_LEN} | extra training data: {USE_EXTRA} | threshold {THRESHOLD}", ""]
for scope, mask in [("All agreements", np.ones(len(real), bool)), ("Indian agreements only", (real.region == "india").values)]:
    y = real.y.values[mask]
    lines += [f"## {scope} ({mask.sum()} clauses, {y.sum()} risky)", "",
              "| System | Precision | Recall | F1 |", "|---|---|---|---|"]
    for name, p in preds.items():
        P, R, F, _ = precision_recall_fscore_support(y, p[mask], average="binary", zero_division=0)
        lines.append(f"| {name} | {P:.2f} | {R:.2f} | {F:.2f} |")
    lines.append("")
report = "\n".join(lines)
print(report)

## Save and download

In [ ]:
out = real[["agreement", "region", "text", "risk", "rule_pred"]].copy()
out["tfidf_prob"] = tfidf_prob.round(4); out["bert_prob"] = bert_prob.round(4); out["fold"] = fold_of
out.to_csv("bert_predictions.csv", index=False)
open("bert_results.md", "w").write(report)
try:
    from google.colab import files
    files.download("bert_results.md"); files.download("bert_predictions.csv")
except Exception:
    print("Saved bert_results.md and bert_predictions.csv")

## Notes for the paper
- Same agreement-grouped splits as `eval_real.py`, so the numbers are directly comparable to Table IV.
- The threshold (0.5) and hyper-parameters are fixed in advance, not tuned on the test folds.
- Results vary slightly between runs and GPUs (about ±0.02). Report the run you keep, with its settings line.
- Rules here are the refined 67-rule version, which was revised after error analysis on these agreements, so the rule and hybrid rows are somewhat optimistic.